In [ ]:
# 1/4 - clone main + prototype (fresh, fault-tolerant)
import subprocess, os
print('===== NATIVE LinearDxFused BITEXACT TEST (PURE TKV, scalar fmaf) =====')
for d,br in [('/content/TVmain','main'),('/content/TVcpu','prototype/cuda-backend')]:
    subprocess.run('rm -rf '+d, shell=True)
    subprocess.run('git clone --depth 1 --branch '+br+' https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
    print(br, subprocess.check_output('git -C '+d+' rev-parse --short HEAD', shell=True).decode().strip())

In [ ]:
# 2/4 - build NATIVE reference tu tkv_linear_dx.c (11 args) + .tkv C .so. NO C injection.
# CUNG flags -O2 -mavx2 -mfma. Khong build tkv_simd_native.c vi no co LinearDxFused 9 args khac.
import subprocess, os
FLAGS='-O2 -mavx2 -mfma'
nroot='/content/TVmain/Release/tools/simd_native'
p=subprocess.run('cd '+nroot+' && gcc '+FLAGS+' -shared -fPIC tkv_linear_dx.c -lm -o linear_dx_native.so', shell=True, capture_output=True, text=True, timeout=600)
print('native linear_dx build rc=',p.returncode)
if p.returncode!=0: print(p.stderr[:2000])
ex='/content/TVcpu/Release/cuda_backend/examples'
cc=ex+'/linear_dx_cpu.cpu.c'; so=ex+'/linear_dx_cpu.so'
txt=open(cc).read()
print('C INJECTED =', 'tkv_ldx' in txt, '(expect False)')
print('fmaf present =', 'fmaf(' in txt, 'isfinite =', '!isfinite(' in txt)
p2=subprocess.run('gcc '+FLAGS+' -shared -fPIC '+cc+' -o '+so+' -lm -lpthread -ldl', shell=True, capture_output=True, text=True, timeout=600)
print('tkv C build rc=',p2.returncode)
if p2.returncode!=0: print(p2.stderr[:2000])

In [ ]:
# 3/4 - bitwise compare LinearDxFused (11 args: gy,w,dx,m,n,k,ndim,T)
import ctypes, numpy as np
PP=ctypes.POINTER(ctypes.c_float); II=ctypes.c_int
args=[PP,II,PP,II,PP,II,II,II,II,II,II]
native=ctypes.CDLL('/content/TVmain/Release/tools/simd_native/linear_dx_native.so').LinearDxFused
native.argtypes=args; native.restype=II
tkv=ctypes.CDLL('/content/TVcpu/Release/cuda_backend/examples/linear_dx_cpu.so').LinearDxFused
tkv.argtypes=args; tkv.restype=II
rng=np.random.default_rng(29)
for m,n,k,ndim,T in [(8,768,3072,3072,256),(8,3072,768,768,256),(16,64,256,256,256),(4,128,512,512,256),(4,64,300,300,256)]:
    gy=rng.standard_normal(m*k).astype(np.float32)
    w=rng.standard_normal(n*ndim).astype(np.float32)
    da=np.zeros(m*n,np.float32); dc=np.zeros(m*n,np.float32)
    rna=native(gy.ctypes.data_as(PP),gy.size,w.ctypes.data_as(PP),w.size,da.ctypes.data_as(PP),da.size,m,n,k,ndim,T)
    rtk=tkv(gy.ctypes.data_as(PP),gy.size,w.ctypes.data_as(PP),w.size,dc.ctypes.data_as(PP),dc.size,m,n,k,ndim,T)
    d=np.count_nonzero(da.view(np.uint32)!=dc.view(np.uint32))
    print('m=%d n=%d k=%d T=%d rc=%d/%d bitwise=%s diff=%d/%d maxabs=%.3g'%(
        m,n,k,T,rna,rtk,d==0,d,da.size,np.abs(da-dc).max()))

In [ ]:
# 4/4 - summary
print('If all bitwise=True -> LinearDxFused pure .tkv (scalar fmaf) reproduces native AVX2 exactly.')